# 04 — Results analysis

Aggregates `eval/results/{base,sft,gptq}/results*.json` into the comparison table that goes in the root `README.md` and `reports/results_summary.md`.

## Load results

In [ ]:
import json
from pathlib import Path

import pandas as pd

LABELS = ["base", "sft", "gptq"]
METRIC_KEYS = {
    "mmlu": "acc,none",
    "arc_challenge": "acc_norm,none",
    "hellaswag": "acc_norm,none",
    "gsm8k": "exact_match,strict-match",
    "truthfulqa_mc2": "acc,none",
    "ifeval": "inst_level_strict_acc,none",
}

rows = {}
for label in LABELS:
    result_files = list(Path(f"eval/results/{label}").rglob("results*.json"))
    if not result_files:
        print(f"WARNING: no results found for '{label}' — run notebook 03 first")
        continue
    with open(result_files[0]) as f:
        data = json.load(f)
    rows[label] = {
        task: data["results"].get(task, {}).get(metric_key)
        for task, metric_key in METRIC_KEYS.items()
    }

df = pd.DataFrame(rows)
df


## Deltas: effect of SFT, effect of quantization

In [ ]:
deltas = pd.DataFrame({
    "delta_sft": df["sft"] - df["base"] if "sft" in df and "base" in df else None,
    "delta_quant": df["gptq"] - df["sft"] if "gptq" in df and "sft" in df else None,
})
deltas


## Plot

In [ ]:
import matplotlib.pyplot as plt

ax = df.T.plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("score")
ax.set_title("Benchmark score by pipeline stage: base -> SFT -> GPTQ 4-bit")
plt.tight_layout()
plt.savefig("reports/figures/stage_comparison.png", dpi=150)
plt.show()


## Next step

Copy the `df` table above into the **Results** section of the root `README.md`, and write the narrative (which benchmarks moved and why) into `reports/results_summary.md`.